# W3.2 — Phân tích baseline

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** cosine_validation.json và run predictions từ W3.1; chạy W3.1 hai lần để kiểm tra tái lập.

Phân bố score giúp tìm overlap positive/negative. Validation hiện cân bằng 1:1,
nên AP không phản ánh trực tiếp prevalence của tất cả cặp node ngoài thực tế.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/runtime.ipynb').as_posix() + '"')
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/metrics.ipynb').as_posix() + '"')

In [ ]:
config = read_json(PROJECT_ROOT/'configs/cosine_baseline.json')
print(config)

{'split_id': 'cora_lp_v1', 'seed': 42, 'feature_preprocessing': 'raw_float64', 'epsilon': 1e-12, 'chunk_size': 256, 'evaluation_split': 'validation', 'atol': 1e-12, 'rtol': 1e-10}


In [ ]:
import matplotlib.pyplot as plt
result = read_json(PROJECT_ROOT/'results/week03/cosine_validation.json')
prediction_path = PROJECT_ROOT/'runs'/result['run_id']/'validation_predictions.npz'
if not prediction_path.exists():
    raise FileNotFoundError(f'Thiếu {prediction_path}; chạy W3.1 để tạo run local.')
with np.load(prediction_path,allow_pickle=False) as archive:
    labels,scores,pairs = archive['labels'],archive['scores'],archive['pairs']
assert lp_metrics(labels,scores) == result['validation']
figdir = PROJECT_ROOT/'report/week03/figures'; figdir.mkdir(parents=True,exist_ok=True)
fig,ax = plt.subplots(figsize=(7,4))
for label,name in [(1,'Positive'),(0,'Negative')]:
    ax.hist(scores[labels==label],bins=np.linspace(0,1,31),alpha=.55,label=name)
ax.set(xlabel='Cosine score',ylabel='Pair count',title='Cora validation: cosine distribution'); ax.legend(); fig.tight_layout()
fig.savefig(figdir/'cosine_score_distribution.png',dpi=130); plt.show()
print({k:result[k] for k in ['run_id','split_id','split_hash','config','validation','elapsed_seconds']})
print('Positive score=0:',int(((labels==1)&(scores==0)).sum()))
print('Highest-scoring negatives:',pairs[:,np.where(labels==0)[0][np.argsort(scores[labels==0])[-5:]]].T.tolist())
print('Reproduction:',result['checks'].get('independent_run_reproduction',{'status':'not_run','reason':'Chạy W3.1 lần thứ hai'}))

{'run_id': '20260928T070135_w3_cosine_a81fd148', 'split_id': 'cora_lp_v1', 'split_hash': '9febb7a31cc7f0362d6c4d39bf7c7a37d6af96a3b9f2ea0f8c7b8630e2cb8c98', 'config': {'split_id': 'cora_lp_v1', 'seed': 42, 'feature_preprocessing': 'raw_float64', 'epsilon': 1e-12, 'chunk_size': 256, 'evaluation_split': 'validation', 'atol': 1e-12, 'rtol': 1e-10}, 'validation': {'roc_auc': 0.8121701918489497, 'average_precision': 0.8248011097161196, 'n_positive': 263, 'n_negative': 263}, 'elapsed_seconds': 0.033440700000937795}
Positive score=0: 29
Highest-scoring negatives: [[1593, 1835], [812, 1277], [1881, 2269], [2154, 2396], [29, 1454]]
Reproduction: {'status': 'passed', 'previous_run_id': '20260928T070124_w3_cosine_5c6533e6', 'atol': 1e-12, 'rtol': 1e-10}


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Phần phân bố overlap nói gì về baseline?
2. AP đổi thế nào khi prevalence thay đổi?
3. Có bằng chứng nào để kết luận GAT tốt hơn chưa?